In [1]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='tqdm')

import sys
sys.path.append('..')  # Go up one level to project root

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from datasets import load_dataset
import pandas as pd
import time
import os

# Import from custom modules
from src.preprocess import build_vocab, SentimentDataset, collate_fn
from src.rnn_mode import RNNClassifier
from src.train import train_epoch
from src.evaluate import get_predictions, calculate_metrics

# Check GPU availability
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device:", torch.cuda.get_device_name(0))
    print("GPU memory:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

# --- Data Loading & Preprocessing ---
print("Loading IMDB dataset...")
dataset = load_dataset("imdb")
train_texts = dataset['train']['text']
train_labels = dataset['train']['label']

print(f"Dataset size: {len(train_texts)} reviews")

# Build vocabulary
VOCAB_SIZE = 20000 
vocab = build_vocab(train_texts, max_size=VOCAB_SIZE)
print(f"Vocabulary size: {len(vocab)}")

# Create full dataset
full_dataset = SentimentDataset(train_texts, train_labels, vocab, max_len=256)

# Split into 70% train, 10% val, 20% test
train_size = int(0.7 * len(full_dataset))
val_size = int(0.1 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size])

print(f"Train set: {len(train_dataset)}, Val set: {len(val_dataset)}, Test set: {len(test_dataset)}")

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)

# Ensure checkpoints directory exists
os.makedirs("../checkpoints", exist_ok=True)

# ============================================================================
# EXPERIMENT 1: RNN Variant Comparison (Vanilla RNN vs LSTM vs GRU)
# Fixed: embed_dim=128, hidden_dim=128, num_layers=1
# ============================================================================
print("\n" + "="*70)
print("EXPERIMENT 1: RNN Variant Comparison")
print("="*70)

exp1_configs = [
    {"name": "Vanilla RNN", "rnn_type": "rnn", "embed_dim": 128, "hidden_dim": 128, "num_layers": 1},
    {"name": "LSTM", "rnn_type": "lstm", "embed_dim": 128, "hidden_dim": 128, "num_layers": 1},
    {"name": "GRU", "rnn_type": "gru", "embed_dim": 128, "hidden_dim": 128, "num_layers": 1},
]

exp1_results = []

for cfg in exp1_configs:
    print(f"\nTraining: {cfg['name']} (embed_dim={cfg['embed_dim']}, hidden_dim={cfg['hidden_dim']}, num_layers={cfg['num_layers']})")
    
    model = RNNClassifier(len(vocab), cfg['embed_dim'], cfg['hidden_dim'], cfg['num_layers'], 2, 
                         rnn_type=cfg['rnn_type'], dropout=0.3).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    best_val_acc = 0.0
    train_losses, train_accs = [], []
    val_losses, val_accs = [], []
    
    start_time = time.time()
    
    for epoch in range(5):
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, is_rnn=True)
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        
        # Validate
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                lengths = (inputs != 0).sum(dim=1).clamp(min=1)
                outputs = model(inputs, lengths)
                val_loss += criterion(outputs, labels).item() * inputs.size(0)
        val_loss /= len(val_dataset)
        val_losses.append(val_loss)
        
        # Get validation accuracy
        val_true, val_pred = get_predictions(model, val_loader, device, is_rnn=True)
        val_acc = calculate_metrics(val_true, val_pred)['Accuracy']
        val_accs.append(val_acc)
        
        print(f"  Epoch {epoch+1}/5 | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f"../checkpoints/rnn_exp1_{cfg['name'].replace(' ', '_')}.pt")
    
    end_time = time.time()
    epoch_time = (end_time - start_time) / 5
    
    # Evaluate on test set
    model.load_state_dict(torch.load(f"../checkpoints/rnn_exp1_{cfg['name'].replace(' ', '_')}.pt"))
    test_true, test_pred = get_predictions(model, test_loader, device, is_rnn=True)
    test_metrics = calculate_metrics(test_true, test_pred)
    
    result = {
        "Config": cfg['name'],
        "Accuracy": test_metrics['Accuracy'],
        "Precision": test_metrics['Precision'],
        "Recall": test_metrics['Recall'],
        "F1": test_metrics['F1 Score'],
        "Time/epoch": epoch_time
    }
    exp1_results.append(result)
    print(f"  Test Accuracy: {test_metrics['Accuracy']:.4f}")

exp1_df = pd.DataFrame(exp1_results)
print("\nExperiment 1 Results:")
print(exp1_df.to_string(index=False))

# ============================================================================
# EXPERIMENT 2: Embedding Dimension (64, 128, 256)
# Using best RNN variant from Experiment 1
# ============================================================================
print("\n" + "="*70)
print("EXPERIMENT 2: Embedding Dimension (using best variant from Exp1)")
print("="*70)

best_variant = exp1_df.loc[exp1_df['Accuracy'].idxmax(), 'Config']
best_rnn_type = exp1_configs[[c['name'] for c in exp1_configs].index(best_variant)]['rnn_type']

exp2_configs = [
    {"name": "embed_64", "rnn_type": best_rnn_type, "embed_dim": 64, "hidden_dim": 128, "num_layers": 1},
    {"name": "embed_128", "rnn_type": best_rnn_type, "embed_dim": 128, "hidden_dim": 128, "num_layers": 1},
    {"name": "embed_256", "rnn_type": best_rnn_type, "embed_dim": 256, "hidden_dim": 128, "num_layers": 1},
]

exp2_results = []

for cfg in exp2_configs:
    print(f"\nTraining: {cfg['name']} (embed_dim={cfg['embed_dim']}, hidden_dim={cfg['hidden_dim']}, rnn_type={cfg['rnn_type']})")
    
    model = RNNClassifier(len(vocab), cfg['embed_dim'], cfg['hidden_dim'], cfg['num_layers'], 2, 
                         rnn_type=cfg['rnn_type'], dropout=0.3).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    best_val_acc = 0.0
    
    start_time = time.time()
    
    for epoch in range(5):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, is_rnn=True)
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                lengths = (inputs != 0).sum(dim=1).clamp(min=1)
                outputs = model(inputs, lengths)
                val_loss += criterion(outputs, labels).item() * inputs.size(0)
        val_loss /= len(val_dataset)
        
        val_true, val_pred = get_predictions(model, val_loader, device, is_rnn=True)
        val_acc = calculate_metrics(val_true, val_pred)['Accuracy']
        
        print(f"  Epoch {epoch+1}/5 | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f"../checkpoints/rnn_exp2_{cfg['name']}.pt")
    
    end_time = time.time()
    epoch_time = (end_time - start_time) / 5
    
    model.load_state_dict(torch.load(f"../checkpoints/rnn_exp2_{cfg['name']}.pt"))
    test_true, test_pred = get_predictions(model, test_loader, device, is_rnn=True)
    test_metrics = calculate_metrics(test_true, test_pred)
    
    result = {
        "Config": cfg['name'],
        "Accuracy": test_metrics['Accuracy'],
        "Precision": test_metrics['Precision'],
        "Recall": test_metrics['Recall'],
        "F1": test_metrics['F1 Score'],
        "Time/epoch": epoch_time
    }
    exp2_results.append(result)
    print(f"  Test Accuracy: {test_metrics['Accuracy']:.4f}")

exp2_df = pd.DataFrame(exp2_results)
print("\nExperiment 2 Results:")
print(exp2_df.to_string(index=False))

# ============================================================================
# EXPERIMENT 3: Number of Layers (1 vs 2)
# Using best embedding dimension from Experiment 2
# ============================================================================
print("\n" + "="*70)
print("EXPERIMENT 3: Number of RNN Layers (using best embed_dim from Exp2)")
print("="*70)

best_embed_config = exp2_df.loc[exp2_df['Accuracy'].idxmax(), 'Config']
best_embed_dim = exp2_configs[[c['name'] for c in exp2_configs].index(best_embed_config)]['embed_dim']

exp3_configs = [
    {"name": "num_layers_1", "rnn_type": best_rnn_type, "embed_dim": best_embed_dim, "hidden_dim": 128, "num_layers": 1},
    {"name": "num_layers_2", "rnn_type": best_rnn_type, "embed_dim": best_embed_dim, "hidden_dim": 128, "num_layers": 2},
]

exp3_results = []

for cfg in exp3_configs:
    print(f"\nTraining: {cfg['name']} (num_layers={cfg['num_layers']}, embed_dim={cfg['embed_dim']}, rnn_type={cfg['rnn_type']})")
    
    model = RNNClassifier(len(vocab), cfg['embed_dim'], cfg['hidden_dim'], cfg['num_layers'], 2, 
                         rnn_type=cfg['rnn_type'], dropout=0.3).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    best_val_acc = 0.0
    
    start_time = time.time()
    
    for epoch in range(5):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, is_rnn=True)
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                lengths = (inputs != 0).sum(dim=1).clamp(min=1)
                outputs = model(inputs, lengths)
                val_loss += criterion(outputs, labels).item() * inputs.size(0)
        val_loss /= len(val_dataset)
        
        val_true, val_pred = get_predictions(model, val_loader, device, is_rnn=True)
        val_acc = calculate_metrics(val_true, val_pred)['Accuracy']
        
        print(f"  Epoch {epoch+1}/5 | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f"../checkpoints/rnn_exp3_{cfg['name']}.pt")
    
    end_time = time.time()
    epoch_time = (end_time - start_time) / 5
    
    model.load_state_dict(torch.load(f"../checkpoints/rnn_exp3_{cfg['name']}.pt"))
    test_true, test_pred = get_predictions(model, test_loader, device, is_rnn=True)
    test_metrics = calculate_metrics(test_true, test_pred)
    
    result = {
        "Config": cfg['name'],
        "Accuracy": test_metrics['Accuracy'],
        "Precision": test_metrics['Precision'],
        "Recall": test_metrics['Recall'],
        "F1": test_metrics['F1 Score'],
        "Time/epoch": epoch_time
    }
    exp3_results.append(result)
    print(f"  Test Accuracy: {test_metrics['Accuracy']:.4f}")

exp3_df = pd.DataFrame(exp3_results)
print("\nExperiment 3 Results:")
print(exp3_df.to_string(index=False))

# ============================================================================
# Save Best Overall RNN Model
# ============================================================================
print("\n" + "="*70)
print("SUMMARY: All RNN Experiments Complete")
print("="*70)

all_results = pd.concat([exp1_df, exp2_df, exp3_df], ignore_index=True)
best_idx = all_results['Accuracy'].idxmax()
best_config = all_results.loc[best_idx]

print(f"\nBest RNN Configuration: {best_config['Config']}")
print(f"Test Accuracy: {best_config['Accuracy']:.4f}")
print(f"Precision: {best_config['Precision']:.4f}, Recall: {best_config['Recall']:.4f}, F1: {best_config['F1']:.4f}")
print(f"Time per epoch: {best_config['Time/epoch']:.2f}s")

# Save best model as rnn_best.pt
if "exp1" in best_config['Config']:
    src = f"../checkpoints/rnn_exp1_{best_config['Config'].replace(' ', '_')}.pt"
elif "exp2" in best_config['Config']:
    src = f"../checkpoints/rnn_exp2_{best_config['Config']}.pt"
else:
    src = f"../checkpoints/rnn_exp3_{best_config['Config']}.pt"

torch.save(torch.load(src), "../checkpoints/rnn_best.pt")
print(f"Best model saved to ../checkpoints/rnn_best.pt")

print("\n" + "="*70)
print("All RNN Experiment Results Summary:")
print("="*70)
print(all_results.to_string(index=False))

c:\Users\Admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.10.0+cu128
CUDA available: True
GPU device: NVIDIA GeForce RTX 5060 Ti
GPU memory: 17.102864384 GB
Using device: cuda

Loading IMDB dataset...
Dataset size: 25000 reviews
Vocabulary size: 20000
Train set: 17500, Val set: 2500, Test set: 5000

EXPERIMENT 1: RNN Variant Comparison

Training: Vanilla RNN (embed_dim=128, hidden_dim=128, num_layers=1)
  Epoch 1/5 | Train Loss: 0.6943 | Train Acc: 0.5403 | Val Acc: 0.5660
  Epoch 2/5 | Train Loss: 0.6638 | Train Acc: 0.6005 | Val Acc: 0.5952
  Epoch 3/5 | Train Loss: 0.6474 | Train Acc: 0.6223 | Val Acc: 0.5824
  Epoch 4/5 | Train Loss: 0.6558 | Train Acc: 0.6067 | Val Acc: 0.5960
  Epoch 5/5 | Train Loss: 0.6352 | Train Acc: 0.6335 | Val Acc: 0.6204
  Test Accuracy: 0.6136

Training: LSTM (embed_dim=128, hidden_dim=128, num_layers=1)
  Epoch 1/5 | Train Loss: 0.6653 | Train Acc: 0.5841 | Val Acc: 0.6648
  Epoch 2/5 | Train Loss: 0.6361 | Train Acc: 0.6373 | Val Acc: 0.5836
  Epoch 3/5 | Train Loss: 0.6349 | Train Acc: 0.6